# 03 — Fused Softmax and Layer Normalization

**Goal:** see how one Triton kernel can perform a sequence of related operations without writing each intermediate result back to global memory.

These examples process one row per program. They are educational implementations; they are not expected to beat every optimized PyTorch implementation.

## 1. Environment

In [ ]:
import math
import statistics
import time
import torch
import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Google Colab, then run this notebook again.")

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Triton: 3.6.0
GPU: Tesla T4


## 2. Row-wise softmax

Softmax converts a row of scores into values that add up to 1. For numerical stability we subtract the row maximum before exponentiation. The Triton kernel performs the max, exponentiation, sum, and division inside one kernel.

In [ ]:
@triton.jit
def row_softmax_kernel(X_ptr, Y_ptr,
                       N_ROWS: tl.constexpr, N_COLS: tl.constexpr,
                       BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N_COLS

    x = tl.load(X_ptr + row * N_COLS + cols, mask=mask, other=-float("inf"))
    row_max = tl.max(x, axis=0)
    exp_x = tl.exp(x - row_max)
    total = tl.sum(exp_x, axis=0)
    y = exp_x / total
    tl.store(Y_ptr + row * N_COLS + cols, y, mask=mask)


def triton_row_softmax(x, out):
    n_rows, n_cols = x.shape
    block_size = triton.next_power_of_2(n_cols)
    row_softmax_kernel[(n_rows,)](x, out, n_rows, n_cols, BLOCK_SIZE=block_size)
    return out

### Verify softmax

In [ ]:
ROWS, COLS = 128, 1024
x = torch.randn((ROWS, COLS), device="cuda", dtype=torch.float32)
out = torch.empty_like(x)
triton_row_softmax(x, out)
expected = torch.softmax(x, dim=1)
torch.testing.assert_close(out, expected, atol=2e-5, rtol=2e-5)
print("Softmax correctness passed")
print("Largest row-sum error:", (out.sum(dim=1) - 1).abs().max().item())

Softmax correctness passed
Largest row-sum error: 1.1920928955078125e-07


## 3. Row-wise layer normalization

Layer normalization subtracts each row's mean and divides by its standard deviation. It then applies learned scale (`gamma`) and shift (`beta`). We use population variance, matching the standard LayerNorm definition.

In [ ]:
@triton.jit
def layer_norm_kernel(X_ptr, Gamma_ptr, Beta_ptr, Y_ptr,
                      N_ROWS: tl.constexpr, N_COLS: tl.constexpr,
                      EPS: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N_COLS

    x = tl.load(X_ptr + row * N_COLS + cols, mask=mask, other=0.0)
    mean = tl.sum(x, axis=0) / N_COLS
    centered = tl.where(mask, x - mean, 0.0)
    variance = tl.sum(centered * centered, axis=0) / N_COLS
    normalized = (x - mean) * tl.rsqrt(variance + EPS)
    gamma = tl.load(Gamma_ptr + cols, mask=mask, other=1.0)
    beta = tl.load(Beta_ptr + cols, mask=mask, other=0.0)
    y = normalized * gamma + beta
    tl.store(Y_ptr + row * N_COLS + cols, y, mask=mask)


def triton_layer_norm(x, gamma, beta, out, eps=1e-5):
    n_rows, n_cols = x.shape
    block_size = triton.next_power_of_2(n_cols)
    layer_norm_kernel[(n_rows,)](
        x, gamma, beta, out, n_rows, n_cols, eps, BLOCK_SIZE=block_size
    )
    return out

### Verify layer normalization

In [ ]:
ROWS, COLS = 128, 1024
x = torch.randn((ROWS, COLS), device="cuda", dtype=torch.float32)
gamma = torch.ones(COLS, device="cuda", dtype=torch.float32)
beta = torch.zeros(COLS, device="cuda", dtype=torch.float32)
out = torch.empty_like(x)
triton_layer_norm(x, gamma, beta, out)
expected = torch.nn.functional.layer_norm(x, (COLS,), gamma, beta, eps=1e-5)
torch.testing.assert_close(out, expected, atol=2e-4, rtol=2e-4)
print("LayerNorm correctness passed")

LayerNorm correctness passed


## 4. Benchmark softmax (basic comparison)

We measure the PyTorch and Triton paths on the same input. The result may depend on GPU model, sizes, compiler, and PyTorch's selected implementation. Record measurements rather than assuming the custom kernel wins.

In [ ]:
def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    """Measure repeated calls with CUDA events; return median ms per call and all rounds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)

    return statistics.median(samples), samples

In [ ]:
ROWS, COLS = 256, 1024
x = torch.randn((ROWS, COLS), device="cuda", dtype=torch.float32)
out_torch = torch.empty_like(x)
out_triton = torch.empty_like(x)

def run_torch():
    torch.softmax(x, dim=1, out=out_torch)

def run_triton():
    triton_row_softmax(x, out_triton)

run_torch(); run_triton()
torch.testing.assert_close(out_triton, out_torch, atol=2e-5, rtol=2e-5)
torch_ms, _ = median_gpu_time_ms(run_torch, repeats=30)
triton_ms, _ = median_gpu_time_ms(run_triton, repeats=30)
print(f"PyTorch softmax: {torch_ms:.6f} ms/call")
print(f"Triton softmax:  {triton_ms:.6f} ms/call")
print(f"PyTorch / Triton ratio: {torch_ms / triton_ms:.3f}x")

PyTorch softmax: 0.015846 ms/call
Triton softmax:  0.020959 ms/call
PyTorch / Triton ratio: 0.756x


## Notes

Fusion can reduce the number of separate kernel launches and avoid storing intermediate tensors in global memory. It is not automatically faster: resource use, row length, and compiler choices matter. Next we apply the online-softmax idea to attention.